In [6]:
from pathlib import Path
import geopandas as gpd
import fiona
import pandas as pd
import os

In [ ]:
def folder_iterator(BASE_DIR, suffix="_v013"):
    """
    Recursively scan BASE_DIR and return all .gpkg files
    ending with the requested suffix.
    """
    files_list = []

    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            if file.lower().endswith(".gpkg"):
                name_without_ext = os.path.splitext(file)[0]

                if name_without_ext.endswith(suffix):
                    gpkg_path = os.path.join(root, file)
                    files_list.append(gpkg_path)

    return files_list


def merge_linestring_layers(gpkg_files, output_shp):
    """
    Iterate over a list of GPKG files, extract all LineString /
    MultiLineString layers, merge them, and write to a shapefile.
    """
    merged = []

    for i, gpkg in enumerate(gpkg_files, start=1):
        print(f"\n[{i}/{len(gpkg_files)}] Processing: {gpkg}")
        
        try:
            layers = fiona.listlayers(gpkg)

            for layer in layers:
                gdf = gpd.read_file(gpkg, layer=layer)

                if gdf.empty:
                    continue

                geom_types = set(gdf.geometry.geom_type.dropna().unique())

                if geom_types.intersection({"LineString", "MultiLineString"}):
                    gdf = gdf.copy()
                    gdf["source_gpkg"] = os.path.basename(gpkg)
                    gdf["source_layer"] = layer
                    merged.append(gdf)

        except Exception as e:
            print(f"Skipping {gpkg}: {e}")

    if not merged:
        raise ValueError("No line layers found.")

    merged_gdf = gpd.GeoDataFrame(
        pd.concat(merged, ignore_index=True),
        crs=merged[0].crs,
    )

    merged_gdf.to_file(output_shp)
    print(f"Saved merged shapefile: {output_shp}")

In [7]:
main_folder = r"Y:\z_resources\ruben\aaaa"
output_shp = r"Y:\z_resources\ruben\aaaa\merged_lines.shp"

gpkg_files = folder_iterator(main_folder, suffix="_v013")

In [ ]:
merge_linestring_layers(gpkg_files, output_shp)